# Silverwing-ML: Full Training Pipeline on Colab GPU

Trains the 102M parameter Silverwing LLM on a T4 GPU using the **real repo**.

Pipeline:
1. **Pretrain** (300 steps) → `experiments/checkpoints/best.pt`
2. **Continued pretrain** (1000 steps) → `experiments/checkpoints/cont/best.pt`
3. **cont2** (2000 steps) → `experiments/checkpoints/cont2/best.pt`
4. **SFT combined** (300 steps) → `experiments/checkpoints/sft-combined/best.pt`
5. Test generation
6. Download checkpoints

**Setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1: Mount Drive
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/silverwing'
os.makedirs(DRIVE, exist_ok=True)
print('Drive mounted:', DRIVE)

In [ ]:
# Cell 2: Install dependencies
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q pyyaml numpy
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - CHANGE RUNTIME!"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Cell 3: Clone real repo (public)
!git clone https://github.com/oledesug-source/silverwing-ml.git /content/Silverwing-ML
os.chdir('/content/Silverwing-ML')
print(f'Working directory: {os.getcwd()}')
!git log --oneline -3
print(f'Corpus: {os.path.exists("experiments/corpus/train.jsonl")}')
print(f'Tokenizer: {os.path.exists("experiments/tokenizer/config.json")}')
print(f'SFT data: {os.path.exists("experiments/sft/sft-v1-combined.jsonl")}')

In [ ]:
# Cell 4: Verify tests pass
!python -m pytest tests/ -q --tb=short 2>&1 | tail -5

## Stage 1: Base Pretrain (300 steps)
Output: `experiments/checkpoints/best.pt`

In [ ]:
# Cell 5: Base pretrain
!python scripts/train.py \
    --config configs/training.yaml \
    --device cuda \
    --batch-size 4 \
    --max-steps 300 \
    --no-clean-repo-check

## Stage 2: Continued pretrain (1000 steps)
Output: `experiments/checkpoints/cont/best.pt`

In [ ]:
# Cell 6: Continued pretrain
!python scripts/train.py \
    --config configs/training_cont.yaml \
    --device cuda \
    --batch-size 4 \
    --no-clean-repo-check

## Stage 3: cont2 (2000 steps)
Output: `experiments/checkpoints/cont2/best.pt`

In [ ]:
# Cell 7: cont2 training
!python scripts/train.py \
    --config configs/training_cont2.yaml \
    --device cuda \
    --batch-size 4 \
    --no-clean-repo-check

## Stage 4: SFT Combined (300 steps)
Output: `experiments/checkpoints/sft-combined/best.pt`

In [ ]:
# Cell 8: SFT combined
!python scripts/train_sft.py \
    --config configs/sft_combined.yaml \
    --device cuda \
    --no-clean-repo-check

## Stage 5: Test Generation

In [ ]:
# Cell 9: Test generation
import torch, json, sys
sys.path.insert(0, '.')
from foundation.model.config import SilverwingConfig
from foundation.model.model import SilverwingDecoder
from foundation.tokenizer import Tokenizer
from foundation.inference import Generator, GenerationConfig

# Load tokenizer
tok = Tokenizer.from_dir('experiments/tokenizer')
print(f'Tokenizer vocab: {tok.vocab_size}')

# Load model from sft-combined checkpoint
ckpt_path = 'experiments/checkpoints/sft-combined/best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_cfg = SilverwingConfig.from_yaml('configs/model.yaml')
model = SilverwingDecoder(model_cfg)
state = torch.load(ckpt_path, map_location=device, weights_only=True)
model.load_state_dict(state['model_state_dict'] if 'model_state_dict' in state else state)
model = model.to(device).eval()
print(f'Model loaded: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

# Generate
gen_cfg = GenerationConfig(max_new_tokens=64, temperature=0.8, top_k=40)
prompts = ['What is 2 + 2? ', 'The answer to 3 * 3 is ', 'Knowledge is ']
for p in prompts:
    output = Generator.generate(model, tok, p, gen_cfg)
    print(f'Prompt: {p!r}')
    print(f'Output: {output!r}')
    print()

## Stage 6: Download Checkpoints

In [ ]:
# Cell 10: Download checkpoints
import zipfile
from pathlib import Path

ckpt_dir = Path('experiments/checkpoints')
files = ['sft-combined/best.pt', 'cont2/best.pt', 'cont/best.pt', 'best.pt']
existing = [f for f in files if (ckpt_dir / f).is_file()]

if not existing:
    print('ERROR: No checkpoint files found!')
else:
    for f in existing:
        mb = (ckpt_dir / f).stat().st_size / 1e6
        print(f'  {f}  ({mb:.0f} MB)')

    zip_path = Path('/content/silverwing-checkpoints.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
        for f in existing:
            zf.write(ckpt_dir / f, f)

    print(f'Created {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)')
    from google.colab import files
    files.download(str(zip_path))
    print('Download started!')
    print('Extract to: F:\\AI\\Silverwing-ML\\experiments\\checkpoints\\')